# Context Aware Nutritional Assessment - Predicting Food Processing Tiers through Machine Learning

### AAI-590 Capstone Project - University of San Diego
### Team Members:
 - Jamshed Nabizada
 - Swapnil Patil

## Objective

This notebook takes the raw Open Food Facts dataset (~10 GB on disk) and applies a systematic cleaning and feature-engineering pipeline to produce a modelling-ready dataset. The pipeline uses **memory-optimised loading** (chunked reading, lean dtypes, aggressive garbage collection) to keep peak RAM usage manageable.

1. **Loads** - reads the dataset in chunks with float32 / category dtypes to avoid the 10 GB peak.
2. **Cleans** - caps outliers, imputes missing nutrients, removes duplicates, drops redundant columns, and standardises labels.
3. **Engineers features** - creates nutrient ratios, additive-presence flags, ingredient-based features, category encodings, and a composite nutritional-profile score.
4. **Selects features** - removes low-variance and highly-correlated columns.
5. **Splits** - produces stratified train / test sets and exports them for downstream modelling.

# Data Cleaning and Feature Engineering

In [ ]:
%pip install -q -r requirements.txt

## 1. Import Libraries and Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gc
import os
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

from src.eda import (
    DEFAULT_LIGHT_DATASET_PATH,
    FULL_DATASET_PATH,
    OpenFoodFactsEDADataLoader,
    print_dataset_overview,
)
from src.cleaning import (
    CLEANED_DATASET_PATH,
    ID_COLS,
    NUTRIENT_COLS_CLEAN,
    RANDOM_STATE,
    TARGET_COL,
    TEST_SIZE,
    CleaningFEPlotter,
    drop_highly_correlated_features,
    drop_low_variance_features,
    run_cleaning_pipeline,
    run_feature_engineering,
)

loader = OpenFoodFactsEDADataLoader()
plotter = CleaningFEPlotter()
DATASET_PATH = FULL_DATASET_PATH

print(f"Dataset path: {DATASET_PATH}")

## 2. Load Raw Dataset

In [ ]:
dataset = loader.load(DATASET_PATH)
df_raw = dataset.df.copy()
NUTRIENT_COLS = dataset.nutrient_cols

# Free the loader's copy early — we work with df_raw from here
del dataset.df
gc.collect()

print_dataset_overview(dataset)
print(f"\nRaw shape: {df_raw.shape}")
df_raw.head(3)

## 3. Data Cleaning Pipeline

The cleaning pipeline runs the following steps in sequence:

| # | Step | Purpose |
|---|------|---------|
| 1 | **Outlier capping** | Clip nutrient values to the 1st–99th percentile range to reduce the influence of anomalous entries. |
| 2 | **Median imputation** | Fill remaining NaN nutrient values with column-level medians (computed post-capping). |
| 3 | **Drop missing target** | Remove rows without a `nova_group` label — they cannot contribute to supervised training. |
| 4 | **Remove duplicates** | Eliminate exact duplicate rows and barcode-level duplicates to prevent data leakage. |
| 5 | **Drop redundant columns** | Remove `sodium_100g` (deterministically derived from `salt_100g`). |
| 6 | **Drop non-modelling columns** | Remove `nutrition_grade_fr` (derived label — potential leakage), `countries_en` and `pnns_groups_2` (unused metadata). |
| 7 | **Standardise labels** | Normalise `nova_group` to canonical integer values. |
| 8 | **Clean text columns** | Replace NaN with empty strings and strip whitespace in free-text fields. |

In [ ]:
rows_before = len(df_raw)
cols_before = df_raw.shape[1]

df_clean, nutrient_cols_clean, clean_log = run_cleaning_pipeline(
    df_raw, NUTRIENT_COLS,
)

# Free the raw dataframe — no longer needed
del df_raw
gc.collect()

rows_after = len(df_clean)
cols_after = df_clean.shape[1]

print("Cleaning Pipeline Log")
display(clean_log)

print(f"\nNutrient columns retained ({len(nutrient_cols_clean)}): {nutrient_cols_clean}")
print(f"Shape after cleaning: {df_clean.shape}")

In [ ]:
plotter.plot_cleaning_summary(rows_before, rows_after, cols_before, cols_after)

In [ ]:
plotter.plot_target_distribution(df_clean)

In [ ]:
# Verify no remaining nulls in nutrient columns
null_counts = df_clean[nutrient_cols_clean].isnull().sum()
print("Remaining nulls in nutrient columns:")
display(null_counts[null_counts > 0] if null_counts.sum() > 0 else "None - all nutrient columns fully populated.")

# Verify no duplicates remain
print(f"\nExact duplicates remaining: {df_clean.duplicated().sum()}")
if "code" in df_clean.columns:
    print(f"Barcode duplicates remaining: {df_clean['code'].duplicated().sum()}")

## 4. Feature Engineering

The feature-engineering pipeline creates new predictive features from the cleaned data:

| Category | Features | Rationale |
|----------|----------|-----------|
| **Nutrient ratios** | sugar/carb, sat-fat/fat, fat-energy %, protein-energy %, fiber/carb, salt/energy, trans-fat/fat | Capture compositional balance rather than absolute values — a key signal for processing detection. |
| **Additive features** | `additives_count`, `has_additives`, top-15 individual additive flags | Additive presence and count are strong indicators of ultra-processing (NOVA 4). |
| **Ingredient features** | `ingredient_count`, `ingredients_length` | Longer ingredient lists often correlate with higher processing complexity. |
| **Palm oil features** | `palm_oil_ingredients`, `has_palm_oil` | Palm oil usage is a strong marker of industrial food formulation (NOVA 4 mean is 55× NOVA 1). |
| **Category encoding** | `pnns_groups_1_encoded` | Encodes food-category context (label-encoded with rare-grouping). |
| **Nutrient profile score** | `nutrient_profile_score` | A composite score balancing negative nutrients (energy, sat fat, trans fat, sugar, salt) against positive ones (fiber, protein). |

In [ ]:
df_fe, all_feature_cols, fe_log = run_feature_engineering(
    df_clean, nutrient_cols_clean,
)

# Free intermediate cleaned dataframe
del df_clean
gc.collect()

print("Feature Engineering Log")
display(fe_log)

engineered_cols = [c for c in all_feature_cols if c not in nutrient_cols_clean]
print(f"\nBase nutrient features : {len(nutrient_cols_clean)}")
print(f"Engineered features   : {len(engineered_cols)}")
print(f"Total feature columns : {len(all_feature_cols)}")

In [ ]:
# Preview the engineered columns
df_fe[engineered_cols].describe().T.round(3)

### 4.1 Engineered Feature Distributions

In [ ]:
# Show distributions for the ratio and composite features
non_binary_engineered = [
    c for c in engineered_cols
    if not c.startswith("add_") and df_fe[c].nunique() > 2
]
plotter.plot_engineered_distributions(df_fe, non_binary_engineered)

### 4.2 Engineered Features by NOVA Group

In [ ]:
plotter.plot_features_by_target(df_fe, non_binary_engineered)

## 5. Feature Selection

To reduce noise and multicollinearity, two automated filters are applied:

1. **Low-variance filter** - drops features whose variance falls below 0.01.
2. **High-correlation filter** - for each pair with |Pearson r| > 0.90, the later column is dropped.

In [ ]:
n_before_sel = len(all_feature_cols)

# 5.1 - Low-variance removal
df_sel, dropped_low_var = drop_low_variance_features(df_fe, all_feature_cols)
remaining_cols = [c for c in all_feature_cols if c not in dropped_low_var]

print(f"Low-variance features dropped ({len(dropped_low_var)}):")
if dropped_low_var:
    print(f"  {dropped_low_var}")
else:
    print("  None")

# 5.2 - High-correlation removal
df_sel, dropped_high_corr = drop_highly_correlated_features(df_sel, remaining_cols)
final_feature_cols = [c for c in remaining_cols if c not in dropped_high_corr]

print(f"\nHighly-correlated features dropped ({len(dropped_high_corr)}):")
if dropped_high_corr:
    print(f"  {dropped_high_corr}")
else:
    print("  None")

n_after_sel = len(final_feature_cols)
print(f"\nFeatures: {n_before_sel} â†’ {n_after_sel}")

In [ ]:
plotter.plot_feature_selection_summary(
    n_before_sel, n_after_sel, dropped_low_var, dropped_high_corr,
)

## 6. Final Dataset Overview

In [ ]:
print(f"Final feature set ({len(final_feature_cols)} features):")
for i, col in enumerate(final_feature_cols, 1):
    print(f"  {i:>2}. {col}")

print(f"\nDataset shape: {df_sel.shape}")
print(f"Target column: {TARGET_COL}")
print(f"Target distribution:")
display(
    df_sel[TARGET_COL]
    .value_counts()
    .sort_index()
    .rename("count")
    .to_frame()
)

In [ ]:
df_sel[final_feature_cols].describe().T.round(3)

### 6.1 Final Correlation Matrix

In [ ]:
plotter.plot_final_correlation(df_sel, final_feature_cols)

## 7. Train Test Split

In [ ]:
X = df_sel[final_feature_cols]
y = df_sel[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train set : {X_train.shape[0]:,} rows  ({100 - TEST_SIZE * 100:.0f}%)")
print(f"Test set  : {X_test.shape[0]:,} rows  ({TEST_SIZE * 100:.0f}%)")
print(f"Features  : {X_train.shape[1]}")

In [ ]:
plotter.plot_split_class_balance(y_train, y_test)

## 8. Export Processed Dataset

In [ ]:
# Combine features and target for export
export_cols = final_feature_cols + [TARGET_COL]

# Optionally retain identifier columns for traceability
id_cols_available = [c for c in ID_COLS if c in df_sel.columns]
if id_cols_available:
    export_cols = id_cols_available + export_cols

df_export = df_sel[export_cols].copy()
df_export.to_csv(CLEANED_DATASET_PATH, index=False)

# Free intermediate dataframes
del df_sel, df_fe
gc.collect()

print(f"Exported cleaned dataset to: {CLEANED_DATASET_PATH}")
print(f"Shape: {df_export.shape}")
print(f"Columns: {list(df_export.columns)}")